# InMemorySaver & Checkpointing

In [2]:
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
checkpointer = InMemorySaver()

In [13]:
config = {"configurable": {"thread_id": "vivek_thread"}}

In [7]:
from pydantic import Field, BaseModel
from typing import Literal
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import HumanMessage

class SeatBookingInput(BaseModel):
    movie_title: str = Field(description="Exact Movie Title")
    seat_count: int = Field(description="Number of seats to book", ge=1, le=10)
    preferred_row: Literal['front', 'middle', 'back'] = Field(
        default='middle',
        description='Preferred seat row'
    )
    
@tool(args_schema=SeatBookingInput)
def book_seats(movie_title: str, seats: int, preferred_row: str, config: str) -> str:
    """Book Seats for a Movie"""
    return f"Booked {seats} seats for {movie_title} in row {preferred_row}"

@tool
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie at the cinema."""
    return "Show is available"

@tool(return_direct=True)
def get_exact_refund_policy() -> str:
    """Tell the refund policy."""
    return "Tickets are refundable up to 2 hours before showtime. No refunds after that."

In [8]:

from langchain.agents import create_agent

cinebot = create_agent(
    model="openai:gpt-5-mini",
    tools=[
        check_showtimes,
        book_seats,
        get_exact_refund_policy
    ],
    system_prompt=(
        "You are CineBot, a friendly cinema booking assistant. "
        "Check showtimes before booking."
    )
)


In [11]:
cinebot.invoke({
    "messages": [
        ("user", "My Name is Vivek")
    ]
})

{'messages': [HumanMessage(content='My Name is Vivek', additional_kwargs={}, response_metadata={}, id='1b12345f-c60f-49cb-aa41-bdc96d1c896e'),
  AIMessage(content='Nice to meet you, Vivek — I’m CineBot. How can I help today?\n\nOptions I can do for you:\n- Check showtimes: tell me the movie title and your city or preferred cinema (and date if not today).\n- Book tickets: tell me the movie title, number of seats (1–10), and preferred row (front/middle/back).\n\nWhat would you like me to do?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 413, 'prompt_tokens': 246, 'total_tokens': 659, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EL4UcFGp2m3kmeG

In [12]:
cinebot.invoke({
    "messages": [
        ("user", "Who am I?")
    ]
})

{'messages': [HumanMessage(content='Who am I?', additional_kwargs={}, response_metadata={}, id='20b30ce1-1027-4f7e-986e-5ce447466480'),
  AIMessage(content='I don’t have any personal information about you—so I can’t tell you who you are unless you tell me. Do you mean:\n\n- “Who am I?” in an existential/psychological sense (want advice on identity)?\n- “Who am I?” in the chat (like your name/account)?\n- Want me to guess with a fun game (I ask yes/no questions)?\n- Want a short personality quiz to help you reflect?\n\nPick one (or tell me more), and I’ll proceed. If you want me to identify you from an account, note I don’t have access to external accounts unless you provide the relevant details.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 394, 'prompt_tokens': 245, 'total_tokens': 639, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tok

In [16]:

from langchain.agents import create_agent

cinebot_stateful = create_agent(
    model="openai:gpt-5-mini",
    tools=[
        check_showtimes,
        book_seats,
        get_exact_refund_policy
    ],
    system_prompt=(
        "You are CineBot, a friendly cinema booking assistant. "
        "Check showtimes before booking."
    ),
    checkpointer=checkpointer
)


In [ ]:
cinebot.invoke({"messages": [("user", "My Name is Vivek")]},config=config)

{'messages': [HumanMessage(content='My Name is Vivek', additional_kwargs={}, response_metadata={}, id='cd8faf83-b820-4df2-b3d8-0161301feb24'),
  AIMessage(content="Hi Vivek — nice to meet you! I'm CineBot. How can I help today?\n\nI can:\n- Check showtimes for a movie,\n- Book seats (up to 10) — I’ll always check showtimes first,\n- Or tell you the refund policy.\n\nWhich movie (and date/time or cinema) would you like me to check?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 273, 'prompt_tokens': 246, 'total_tokens': 519, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EL4atv4JgLJLSYgravVkdVbBVqu3f', 'service_tier': 'default', 'finish_reason':

In [19]:
cinebot.invoke({"messages": [("user", "Who am I?")]},config=config)

{'messages': [HumanMessage(content='My Name is Vivek', additional_kwargs={}, response_metadata={}, id='cd8faf83-b820-4df2-b3d8-0161301feb24'),
  AIMessage(content="Hi Vivek — nice to meet you! I'm CineBot. How can I help today?\n\nI can:\n- Check showtimes for a movie,\n- Book seats (up to 10) — I’ll always check showtimes first,\n- Or tell you the refund policy.\n\nWhich movie (and date/time or cinema) would you like me to check?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 273, 'prompt_tokens': 246, 'total_tokens': 519, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EL4atv4JgLJLSYgravVkdVbBVqu3f', 'service_tier': 'default', 'finish_reason':